In [1]:
cd(joinpath(pwd(), "src/LorentzianSimplexSolver"))

using Pkg
Pkg.activate(".")
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/src/LorentzianSimplexSolver`


In [2]:
include("../../scripts/run_geometry.jl");
include("../../scripts/run_action.jl");
include("../../scripts/run_dlogEh_dX.jl")
include("../../scripts/run_deta_dl.jl")
include("../perturbations/TransverseBasis.jl")
include("../perturbations/Soln_dY_dX.jl")
include("../perturbations/DθDl.jl");

In [3]:
using JLD2

using .RunGeometry
using .RunAction
using .RunDlogEhDX
using .DηDLUtils
using .TransverseBasis
using .Soln_dY_dX

#### Geometry setup

In [ ]:
simplices = [[1,2,3,4,6],[1,2,3,5,6],[1,2,4,5,6],[1,3,4,5,6],[2,3,4,5,6]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

coords_lines = [
    "0, 0, 0, 0",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "-0.068,-0.27,-0.5,-1.3",
]

const ScalarT = Float64
# tol = 1e-10;
# const ScalarT = BigFloat
const tol = parse(ScalarT, "1e-8")

if ScalarT === BigFloat
    setprecision(BigFloat, 80)
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(80)
    # LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
end

gamma_vals = ScalarT(1.0);

vertex_coords = Dict{Int, Vector{ScalarT}}()  

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [5]:
geom = run_geometry_pipeline(simplices, coords_lines, ScalarT, tol);

#### Action and variables calculation

In [6]:
deficit_angles, dihedral_angles, _, _, iRegge = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom, simplices, vertex_coords);

In [7]:
γ = LorentzianSimplexSolver.DefineAction.γsym()
sd, S_symbols, phase_soln = RunAction.run_action(geom, dihedral_angles, γ);

g_vars = geom.varias[:g_var]
z_vars = geom.varias[:z_var]
η_vars = geom.varias[:η_var]

vars = vcat(g_vars, z_vars, η_vars);

In [8]:
using SymEngine
vals = LorentzianSimplexSolver.ActionEvaluation.build_value_dict(sd, γ; γval=gamma_vals);
S = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S_symbols, phase_soln);
S_val = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S, vals);
SF_action = SymEngine.expand(S_val)
@show SF_action, iRegge;

(SF_action, iRegge) = (6.37226121460518e-14 - 0.0962834958395493*im, 0.0 - 0.09628349583654977im)


#### Computation of $\frac{\partial \log E_h}{\partial X^\alpha}$

In [9]:
nh = length(geom.connectivity[1]["OrderBulkFaces"])

dlogEh_dX_sym = RunDlogEhDX.run_dlogEh_dX(geom)

dlogEh_dX_vals = RunDlogEhDX.evaluate_dlogEh_dX(dlogEh_dX_sym, geom, sd; γval=gamma_vals);

#### Computation of $\frac{\partial \log E_b}{\partial X^\alpha}$, $\frac{\partial \log E_b}{\partial Y^\alpha}$

In [10]:
nb = length(geom.connectivity[1]["OrderBDryFaces"])

dlogEb_dX_sym, dlogEb_dY_sym, Y_vars = RunDlogEhDX.run_dlogEb_dXY(geom);

dlogEb_dX_vals, dlogEb_dY_vals = RunDlogEhDX.evaluate_dlogEb_dXY(dlogEb_dX_sym, dlogEb_dY_sym, Y_vars, geom, sd, phase_soln; γval=gamma_vals);

#### Computation of $\frac{\partial \log k_b E_b}{\partial X^\alpha}$, $\frac{\partial \log k_b E_b}{\partial Y^\alpha}$, $\frac{\partial^2 \log k_b E_b}{\partial X^\alpha \partial Y^\alpha}$ and $\frac{\partial^2 \log k_b E_b}{\partial Y^\alpha}$

In [11]:
dkbEb_dX_sym, dkbEb_dY_sym, d2kbEb_dXdY_sym, d2kbEb_dYdY_sym = RunDlogEhDX.run_kblogEb_dXY(geom, Y_vars);

In [12]:
dkbEb_dX_vals, dkbEb_dY_vals, d2kbEb_dXdY_vals, d2kbEb_dYdY_vals = RunDlogEhDX.evaluate_kblogEb_all(dkbEb_dX_sym, dkbEb_dY_sym, d2kbEb_dXdY_sym, d2kbEb_dYdY_sym, geom, sd, Y_vars, phase_soln; γval=gamma_vals);

In [13]:
d2kbEb_dXdY_vals_sumb = sum(d2kbEb_dXdY_vals[i, :, :] for i in 1:nb);

#### Computation of $\frac{\partial \eta_h}{\partial \ell_s}$

In [14]:
# dηdl_matrix is a nh x nl matrix
η_h_vertices = DηDLUtils.get_bulk_faces_vertices(geom)

bulk_edges, bdry_edges = DηDLUtils.get_bulk_edges(geom, η_h_vertices)
bdry_edges_perturb = [bdry_edges[1]] # here we only perturb one boundary edge
perturb_edges = vcat(bulk_edges, bdry_edges_perturb)
    
dηdl_matrix = DηDLUtils.build_dηdl_matrix(η_h_vertices, perturb_edges, vertex_coords, ScalarT, gamma_vals);
nl = length(perturb_edges)
nt = nh - nl;

#### Computation of $\hat{e}^i_h$

In [15]:
# eListHT is a nh x nt matrix
eListHT = TransverseBasis.compute_transverse_basis(dηdl_matrix, tol);

#### Computation of $\frac{\partial k_b}{\partial \ell_s}$ and $\frac{\partial^2 k_b}{\partial \ell_s^2}$

In [16]:
dkbdl, d2kb_dldl = Soln_dY_dX.dkb_dl(geom, nl, bdry_edges_perturb, vertex_coords ;γ=gamma_vals);

#### computation of $\delta\epsilon$ and $\delta\Theta$

In [17]:
DϵDl = DθDl_module.compute_dθDl(simplices, η_h_vertices, perturb_edges, vertex_coords, geom.connectivity[1]["Tets"], ScalarT);

In [18]:
bd_faces = geom.connectivity[1]["OrderBDryFaces"];
kb_vertices = [geom.connectivity[1]["TetFaces"][f[1][1]][f[1][2]][f[1][3]] for f in bd_faces];
DΘDl = DθDl_module.compute_dθDl(simplices, kb_vertices, perturb_edges, vertex_coords, geom.connectivity[1]["Tets"], ScalarT);

#### Hessian matrix computation or read Hessian from files

In [19]:
H_symbols = LorentzianSimplexSolver.EOMsHessian.compute_Hessian_block_half(S, vars);
H_eval = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian_block(H_symbols, sd; γ=gamma_vals);

#### Computation of $B_\alpha$ matrix, $C_\alpha$, $\frac{\partial Y}{\partial \ell}$ matrix and $\frac{\partial X}{\partial \ell}$ matrix

In [20]:
ng = length(g_vars)
nz = length(z_vars)
nX = ng + nz
X_vars = vcat(g_vars, z_vars)

invHessianXX = inv(H_eval[1:nX, 1:nX]);

In [21]:
Bα = transpose(transpose(dηdl_matrix) * dlogEh_dX_vals);

In [22]:
M_matrix = vcat(dlogEb_dY_vals[:, nb+1:end] - dlogEb_dX_vals * invHessianXX * d2kbEb_dXdY_vals_sumb[:, nb+1:end], -dlogEh_dX_vals * invHessianXX * d2kbEb_dXdY_vals_sumb[:, nb+1:end]);
dmatrix = vcat(im * gamma_vals/2 * DΘDl + dlogEb_dX_vals * invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl), im * gamma_vals/2 * DϵDl + dlogEh_dX_vals * invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl));
using LinearAlgebra
DξDl = pinv(M_matrix, atol=tol, rtol=tol) * dmatrix;

In [23]:
DYDl = vcat(dkbdl, DξDl);

In [24]:
DXDl = -invHessianXX * (Bα + d2kbEb_dXdY_vals_sumb[:, 1:nb] * dkbdl + d2kbEb_dXdY_vals_sumb[:, nb+1:end] * DξDl);

In [25]:
Cα =d2kbEb_dXdY_vals_sumb * DYDl;

#### Computation of $\delta^2 Y$ matrix

In [26]:
kb_vals = [ScalarT(subs(Y_vars[i], vals)) for i in 1:nb];

Mm = [kb_vals[b] * dlogEb_dY_vals[b, j] for b in 1:nb, j in nb+1:length(Y_vars)];
R = [dkbdl[i, j] *  transpose(dlogEb_dY_vals[i, nb+1:end]) * DξDl[:, j] + transpose(DXDl[:, j]) * d2kbEb_dXdY_vals[i, :, nb+1:end] * DξDl[:, j] + transpose(DξDl[:, j]) * d2kbEb_dYdY_vals[i, nb+1:end, nb+1:end] * DξDl[:, j] for i in 1:nb, j in 1:nl]

D2ξDl2 = - pinv(Mm, atol=tol, rtol=tol) * R[:, end];

In [27]:
nxi = length(Y_vars) - nb
D2ξDl2_matrix = [j==k==nl ? D2ξDl2[b] : 0 for b in 1:nxi, j in 1:nl, k in 1:nl]
D2YDl2_matrix = vcat(d2kb_dldl, D2ξDl2_matrix);

#### Computation of boundary linear and quadratic term $\mathcal{I}_{bdry}^{(1)}$ and $\mathcal{I}_{bdry}^{(2)}$

In [28]:
BA = vcat(Bα + Cα, zeros(nt, nl));

In [29]:
eta_h = [LorentzianSimplexSolver.DefineSymbols.make_symbol("η_$(faces[1][1])$(faces[1][2])$(faces[1][3])") for faces in geom.connectivity[1]["OrderBulkFaces"]]
eta_h_vals = [ScalarT(subs(eta_h[i], vals)) for i in 1:nh];
Ahh = Matrix(Diagonal(eta_h_vals))

hαβ = H_eval[1:nX, 1:nX] + transpose(dlogEh_dX_vals) * Ahh * dlogEh_dX_vals;
Hαi = transpose(dlogEh_dX_vals) *  eListHT;

HIJ = [hαβ Hαi; transpose(Hαi) zeros(nt, nt)];
invHIJ = inv(HIJ);

In [30]:
DtXDl = -invHIJ * BA;

In [31]:
d2kbEb_dYdY_vals_sumb = sum(d2kbEb_dYdY_vals[i, :, :] for i in 1:nb);
dkbEb_dY_vals_sumb = sum(dkbEb_dY_vals[i, :] for i in 1:nb);
Iboundary_linear = transpose(dkbEb_dY_vals_sumb) * DYDl

1×6 transpose(::Vector{Any}) with eltype Any:
 -8.60235911707644e-09 - 1.53385726086427e-09*im  …  -9.01454050294959e-11 - 1.08784066305548*im

In [32]:
Iboundary_qadratic = 1/2 * transpose(DYDl) * d2kbEb_dYdY_vals_sumb * DYDl + 1/2 * sum(dkbEb_dY_vals_sumb[i] * D2YDl2_matrix[i, :, :] for i in 1:length(Y_vars))

6×6 Matrix{Basic}:
 -3.87400934893387e-17 - 5.91432703965183e-17*im  …   2.92688223001586e-09 - 2.95881727974918e-09*im
  5.75518634971717e-19 + 6.94542886037857e-19*im     -3.58008407590147e-11 + 3.52554492876003e-11*im
 -2.35040923612798e-17 - 3.61890003731562e-17*im      1.78938044997035e-09 - 1.80032167181711e-09*im
     -2.27236186681e-17 - 3.3998103030805e-17*im      1.68686968072965e-09 - 1.72057427936064e-09*im
 -3.73866681746714e-18 - 6.27858594479473e-18*im      3.05698995499732e-10 - 2.92958899300044e-10*im
  2.92688223001586e-09 - 2.95881727974918e-09*im  …        -7.78095311924892 + 0.0692844161730779*im

#### Computation of Spinfoam quadratic term $(S_{eff} - S^{(0)}_{eff})^{(2)}$

In [33]:
SF_quadratic = transpose(BA) * DtXDl + 1/2 * transpose(DtXDl) * HIJ * DtXDl + Iboundary_qadratic

6×6 Matrix{Basic}:
  -47.0782494278839 - 1.62683299432073*im  …    2.99865895974059 + 0.103621451919647*im
 -5.30433554148642 - 0.183296281718031*im     0.337860764704404 + 0.0116750952980153*im
  -32.6886135367472 - 1.12958565123243*im      2.08210808798387 + 0.0719491846484171*im
 -16.0980676102308 - 0.556283801011573*im      1.02536978654637 + 0.0354326037012782*im
 -12.8500541276117 - 0.444045653453528*im      0.818486891258841 + 0.028283579228849*im
  2.99865895976691 + 0.103621451938878*im  …   -0.191000211785885 - 2.76297359337925*im

In [34]:
SF_quadratic_form2 = -1/2 * transpose(BA[1:nX, :]) * invHIJ[1:nX, 1:nX] * BA[1:nX, :] + Iboundary_qadratic

6×6 Matrix{Basic}:
  -47.0782494278761 - 1.62683299430123*im  …    2.99865895974226 + 0.103621451918368*im
  -5.30433554148989 - 0.18329628170997*im     0.337860764704242 + 0.0116750952958039*im
  -32.6886135367314 - 1.12958565121718*im      2.08210808798419 + 0.0719491846445789*im
  -16.098067610203 - 0.556283801006532*im      1.02536978654653 + 0.0354326037055391*im
 -12.8500541276148 - 0.444045653454155*im     0.818486891256876 + 0.0282835792258584*im
  2.99865895975434 + 0.103621451927471*im  …   -0.191000211785712 - 2.76297359337712*im

In [35]:
SF_linear_bdry = Iboundary_linear

1×6 transpose(::Vector{Any}) with eltype Any:
 -8.60235911707644e-09 - 1.53385726086427e-09*im  …  -9.01454050294959e-11 - 1.08784066305548*im

#### Computation of Spinfoam quadratic term from deficit angles $\delta\epsilon$

In [36]:
ω = - dlogEh_dX_vals * invHessianXX * transpose(dlogEh_dX_vals)
ρ = inv(inv(Ahh) - ω);

In [37]:
Sij = -transpose(eListHT) * dlogEh_dX_vals * inv(hαβ) * transpose(dlogEh_dX_vals) * eListHT
κ = eListHT * inv(Sij) * transpose(eListHT)

M_kernel =  ρ - ρ * inv(Ahh) * κ * inv(Ahh) * ρ;

In [38]:
correction_term = - gamma_vals^2/8 * transpose(DϵDl) * M_kernel * DϵDl

6×6 Matrix{ComplexF64}:
 -47.0782+834.859im   -5.30434+94.0641im  …   2.99866-53.1765im
 -5.30434+94.0641im  -0.597643+10.5983im     0.337861-5.99143im
 -32.6886+579.681im   -3.68305+65.313im       2.08211-36.9229im
 -16.0981+285.474im   -1.81378+32.1645im      1.02537-18.1833im
 -12.8501+227.876im   -1.44782+25.6749im     0.818487-14.5146im
  2.99866-53.1765im   0.337861-5.99143im  …    -0.191+3.38709im

In [44]:
correction_term

6×6 Matrix{ComplexF64}:
 -47.0782+834.859im   -5.30434+94.0641im  …   2.99866-53.1765im
 -5.30434+94.0641im  -0.597643+10.5983im     0.337861-5.99143im
 -32.6886+579.681im   -3.68305+65.313im       2.08211-36.9229im
 -16.0981+285.474im   -1.81378+32.1645im      1.02537-18.1833im
 -12.8501+227.876im   -1.44782+25.6749im     0.818487-14.5146im
  2.99866-53.1765im   0.337861-5.99143im  …    -0.191+3.38709im

In [39]:
iSRegge_quadratic = im*(gamma_vals/4 * transpose(dηdl_matrix) * DϵDl + gamma_vals/4 * (transpose(dkbdl) * DΘDl + sum(dihedral_angles[i] * d2kb_dldl[i, :, :] for i in 1:nb)))

6×6 Matrix{Basic}:
 -0.0 - 836.485606094484*im  …   0.0 + 53.2801260775125*im
 -0.0 - 94.2473516886146*im      0.0 + 6.00310482793063*im
 -0.0 - 580.810778633642*im      0.0 + 36.9948643315728*im
 -0.0 - 286.030215754369*im      0.0 + 18.2187545683246*im
 -0.0 - 228.319562566728*im      0.0 + 14.5428624125565*im
  0.0 + 53.2801260775138*im  …  -0.0 - 6.15006220467917*im

In [40]:
SF_quadratic_wrt_deficit = iSRegge_quadratic + correction_term

6×6 Matrix{Basic}:
  -47.0782494267889 - 1.62683299428761*im  …    2.99865896877705 + 0.103621468687159*im
 -5.30433554180037 - 0.183296281764228*im     0.337860764567249 + 0.0116750950998066*im
  -32.6886135362789 - 1.12958565123097*im      2.08210809346008 + 0.0719491949080506*im
 -16.0980676090201 - 0.556283800915594*im      1.02536979185761 + 0.0354326133423299*im
 -12.8500541279081 - 0.444045653543839*im     0.818486892122963 + 0.0282835810082922*im
  2.99865896877711 + 0.103621468688445*im  …   -0.191000211785924 - 2.76297359337814*im

In [41]:
iSRegge_linear = im/2 * gamma_vals * transpose(dkbdl) * dihedral_angles

6-element Vector{Any}:
                         0
                         0
                         0
                         0
                         0
 0.0 - 1.08784066308672*im

In [42]:
sum(iSRegge_linear - transpose(SF_linear_bdry))

1.9645104305905e-08 + 3.54779329534945e-09*im

In [43]:
sum(SF_quadratic_wrt_deficit - SF_quadratic_form2)

4.51152783742526e-08 + 7.62405299257335e-08*im